# EL (Extract and Load) MinIO Landing to MinIO Bronze, From Parquet to Delta format

### Install python dotenv for get the environment variables

In [1]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


## Import Libs

In [2]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
import logging
from configurations import configurations as config_file # Import configurations.py from the configurations folder
from functions import functions as func_file # Import functions.py from the functions folder

# Import for get the environment variables 
from dotenv import load_dotenv
import os

### Import environment variables

In [3]:
load_dotenv()

MINIO_USER=os.getenv("MINIO_USER")
MINIO_PASSWORD=os.getenv("MINIO_PASSWORD")
MINIO_CONTAINER=os.getenv("MINIO_CONTAINER")
POSTGRES_USER=os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD=os.getenv("POSTGRES_PASSWORD")
POSTGRES_CONTAINER=os.getenv("POSTGRES_CONTAINER")

In [4]:
conf = SparkConf()

conf.setAppName("Extract and Load minIO Landing to MinIO Bronze, From Parquet to Delta Format") # Spark application name, Usefull for logs
# Add the jars from hadoop-aws and aws-java-sdk-bundle is necessary for org.apache.hadoop.fs.s3a.S3AFileSystem,
# add the Postgresql JDBC jar is necessary for connect on database. Add the delta-spark is necessary for delta catalog, all this Jars is auto-download from spark
conf.set("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,"
         "com.amazonaws:aws-java-sdk-bundle:1.12.767,"
         "org.postgresql:postgresql:42.7.2,"
         "io.delta:delta-spark_2.12:3.2.0")
conf.set("spark.hadoop.fs.s3a.endpoint",f"http://{MINIO_CONTAINER}:9000") # Container and Port from MinIO
conf.set("spark.hadoop.fs.s3a.access.key", MINIO_USER) # Login from MinIO
conf.set("spark.hadoop.fs.s3a.secret.key", MINIO_PASSWORD) # Password from MinIO
conf.set("spark.hadoop.fs.s3a.path.style.access", True) # Enforces the use of URLs as the format. Without this, Spark attempts to use the AWS standard (bucket.endpoint), which fails in MinIO
conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # Talk to Hadoop/Spark to use new conector S3A
conf.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") # How to credentials are acess via config(access key + secret)
conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") # active extension from Delta Lake
conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") # Change the standard catalog from spark to Delta 
conf.set("hive.metastore.uris", "thrift://metastore:9083") # Connect to Hive Metastore external

spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()


In [5]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

logging.info("Starting convertions from Minio landing (Parquet) to Minio Bronze (Delta)...")

2026-05-04 21:31:38,669 - INFO - Starting convertions from Minio landing (Parquet) to Minio Bronze (Delta)...


In [7]:
for table in config_file.tables_postgres_adventureworks.values():
    
    # Switching the dot(.) for Underline(_) on table name 
    table_name = func_file.convert_table_name(table)
    
    # Path from landing tables 
    table_path = config_file.data_lakehouse_path['landing']

    # Output from bronze layer
    output_path = config_file.data_lakehouse_path['bronze']

    try:
        # Logging the inital process
        logging.info(f"processing table {table_name}")
        
        # Reading data from minio landing
        logging.info(f"Reading table {table_name}...")
        dataframe = spark.read.format("parquet").load(f"{table_path}{table_name}")
        
        # Adding a data from last update on Minio Bronze
        df_with_data = func_file.add_data_last_update(dataframe)

        # Writing the table in format delta to the bronze layer
        logging.info(f"Writing table {table_name}...")
        df_with_data.write.format("delta").mode("overwrite").partitionBy("month_key").save(f"{output_path}bronze_{table_name}")

        # Logging process completed
        logging.info(f"Write table {table_name} Completed and saved on: {output_path}bronze_{table_name}")
        
    except Execption as e:

        logging.info(f"Error processing {table_name}: {str(e)}")

logging.info(f"Ingestions to bronze layer Completed")

2026-05-04 21:32:23,402 - INFO - processing table sales_countryregioncurrency
2026-05-04 21:32:23,403 - INFO - Reading table sales_countryregioncurrency...
2026-05-04 21:32:29,208 - INFO - Read table sales_countryregioncurrency Completed
2026-05-04 21:32:29,396 - INFO - Writing table sales_countryregioncurrency...
2026-05-04 21:32:43,511 - INFO - Write table sales_countryregioncurrency Completed
2026-05-04 21:32:43,512 - INFO - processing table sales_creditcard
2026-05-04 21:32:43,513 - INFO - Reading table sales_creditcard...
2026-05-04 21:32:45,045 - INFO - Read table sales_creditcard Completed
2026-05-04 21:32:45,065 - INFO - Writing table sales_creditcard...
2026-05-04 21:32:49,808 - INFO - Write table sales_creditcard Completed
2026-05-04 21:32:49,810 - INFO - processing table sales_currency
2026-05-04 21:32:49,811 - INFO - Reading table sales_currency...
2026-05-04 21:32:50,010 - INFO - Read table sales_currency Completed
2026-05-04 21:32:50,020 - INFO - Writing table sales_curre